# H3 - Floors vs Lot Size

**Hypothesis (William Rodriguez - City House):**
Houses in more central, city-like areas tend to have more floors but smaller lot sizes. This compact, vertical style is what William's city house should look like.

**Research question:** Do houses with more floors tend to sit on smaller lots?

Since William is looking for a **city house**, we restrict this analysis to **Urban** houses from the very beginning (Section 1.1), rather than analyzing all houses and filtering down at the end. We then clean the data thoroughly (Section 3) and test the floors-vs-lot-size relationship on that Urban subset (Section 4).

## 1. Understanding the Data

We load the dataset, restrict it to Urban houses (the population relevant to William's city house search - see 1.1), and take a first look at the two columns relevant to H3.

In [1]:
# import the libraries we need for this notebook
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

# load the dataset
df = pd.read_csv("../data/eda.csv")

# show the first few rows to get an overview
df.head()

,date,price,house_id,id
0,2014-10-13,221900.0,7129300520,1
1,2014-12-09,538000.0,6414100192,2
2,2015-02-25,180000.0,5631500400,3
3,2014-12-09,604000.0,2487200875,4
4,2015-02-18,510000.0,1954400510,5


### 1.1 Restricting to Urban Houses (RUCA Codes)

William wants a **city house**, so before exploring anything we narrow the dataset down to Urban houses only, using the USDA / WWAMI **Rural-Urban Commuting Area (RUCA) codes** - an official, published zip-code-level classification, instead of a subjective guess.

**Why RUCA?**
- It's an external, citable, reproducible source instead of a hardcoded list.
- It's published at zip-code granularity, so it joins directly onto our `zipcode` column.

**What we found:** all 70 zip codes in this dataset are RUCA 1 ("Metropolitan area core") or RUCA 2 ("Metropolitan area high commuting") - King County has no zip codes at the standard national "rural" threshold (RUCA 3+), since it sits entirely inside the Seattle metro area. To still get a meaningful split:
- **RUCA 1** (commutes stay inside the urban area) -> **Urban**
- **RUCA 2** (30%+ commute out to the urban core, i.e. exurban) -> **Rural**

Steps:
1. Load the RUCA zip-code lookup table.
2. Join it onto our data by `zipcode`.
3. Classify RUCA 1 -> Urban, RUCA 2 -> Rural.
4. Keep only the Urban houses for the rest of this notebook.

In [2]:
# Steps 1-3: load the RUCA zip-code lookup table, join it onto our data by
# zipcode, and classify each house as Urban or Rural.
#
# Source: USDA Economic Research Service RUCA codes, ZIP-code approximation,
# published by the WWAMI Rural Health Research Center, University of Washington
# (https://depts.washington.edu/uwruca/ruca-download.php), 2006 ZIP-code version.
ruca = pd.read_csv("../data/ruca_zipcodes.csv")

df = df.merge(ruca[["zipcode", "ruca_primary"]], on="zipcode", how="left")

# check the join worked: every house should have found a matching RUCA code
missing_ruca = df["ruca_primary"].isna().sum()
print(f"Houses with no matching RUCA code: {missing_ruca}")

# RUCA 1 = Metropolitan area core (commutes stay inside the urban area) -> Urban
# RUCA 2 = Metropolitan area high commuting (30%+ commute out to the core) -> Rural
df["location_type"] = df["ruca_primary"].apply(lambda x: "Urban" if x == 1 else "Rural")
print(df["location_type"].value_counts())

KeyError: 'zipcode'

In [ ]:
# Step 4: keep only Urban houses - William is looking for a city house, so
# Rural houses are out of scope for the rest of this notebook.
rows_before = len(df)
df = df[df["location_type"] == "Urban"].copy()

print(f"Kept {len(df):,} Urban houses out of {rows_before:,} total ({len(df) / rows_before:.1%}).")

In [ ]:
# quick summary statistics for the two columns we care about in H3
df[["floors", "sqft_lot"]].describe()

In [ ]:
# check for missing values in the columns we need for H3
df[["floors", "sqft_lot"]].isna().sum()

## 2. Exploring the Data

Before testing anything, we look at how `floors` and `sqft_lot` are distributed for these Urban houses.

In [ ]:
# floors is a discrete/categorical-like value (1, 1.5, 2, ...)
# a bar chart of value counts shows how common each floor count is
df["floors"].value_counts().sort_index().plot(kind="bar")
plt.title("Number of Houses per Floor Count")
plt.xlabel("Floors")
plt.ylabel("Number of Houses")
plt.show()

In [ ]:
# sqft_lot is continuous, so a histogram is a good way to see its distribution
df["sqft_lot"].plot(kind="hist", bins=50)
plt.title("Distribution of Lot Size (sqft_lot)")
plt.xlabel("Lot Size (sqft)")
plt.ylabel("Number of Houses")
plt.show()

The histogram is heavily skewed to the right: most houses have a small lot, but a few houses have a very large lot and stretch out the x-axis. These extreme values are outliers we should handle before comparing groups.

## 3. Cleaning the Data

Beyond a single "drop the top 1%" cutoff, we apply a more thorough cleaning pass, computed on the Urban-only subset (so the bounds reflect city lot sizes, not the whole county):

1. **IQR-based outlier removal on `sqft_lot` (both tails).** Instead of an arbitrary percentile cutoff, we use the interquartile range method (`Q1 - 1.5*IQR` to `Q3 + 1.5*IQR`) - the same statistically standard approach already used in the H1 notebooks. This also removes unrealistically *tiny* lots, not just unusually huge ones.
2. **Duplicate house check.** If the same house (`id`) was sold more than once, it would be counted multiple times in our analysis and could distort the correlation. We check for duplicate `id`s and, if any exist, keep only the most recent sale per house.
3. **`floors` sanity check.** We confirm `floors` only contains the expected discrete values (between 1 and 3.5), so no invalid data slips through unnoticed.

In [ ]:
# 3.1 Remove sqft_lot outliers using the IQR method (both tails), computed
# on the Urban-only data
Q1 = df["sqft_lot"].quantile(0.25)
Q3 = df["sqft_lot"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean = df[(df["sqft_lot"] >= lower_bound) & (df["sqft_lot"] <= upper_bound)].copy()

rows_dropped = len(df) - len(df_clean)
print(f"IQR bounds for sqft_lot: [{lower_bound:,.0f}, {upper_bound:,.0f}]")
print(f"Dropped {rows_dropped} rows ({rows_dropped / len(df):.1%} of the Urban data) as sqft_lot outliers.")

# note: sqft_lot is heavily right-skewed, so the IQR lower bound comes out
# negative - since a lot size can't be negative, this rule ends up trimming
# only the upper tail in practice. We keep the two-sided formula anyway since
# it is the standard, unbiased method and would trim the lower tail too if
# the data were less skewed.

In [ ]:
# sqft_lot is continuous, so a histogram is a good way to see its distribution
df_clean["sqft_lot"].plot(kind="hist", bins=50)
plt.title("Distribution of Lot Size (sqft_lot) after removing outliers")
plt.xlabel("Lot Size (sqft)")
plt.ylabel("Number of Houses")
plt.show()

In [ ]:
# 3.2 Check for duplicate house sales (the same house id appearing more than once)
duplicate_count = df_clean["id"].duplicated().sum()
print(f"Duplicate house ids found: {duplicate_count}")

if duplicate_count > 0:
    # keep only the most recent sale per house
    df_clean = df_clean.sort_values("date").drop_duplicates(subset="id", keep="last")
    print(f"Kept {len(df_clean):,} rows after removing duplicate house sales.")

In [ ]:
# 3.3 Sanity-check floors: confirm only the expected discrete values are present
print("Unique floors values:", sorted(df_clean["floors"].unique()))
assert df_clean["floors"].between(1, 3.5).all(), "Found floors values outside the expected 1-3.5 range" 

## 4. Relationships in the Data

Now we look at how `floors` and `sqft_lot` relate to each other for these Urban houses, using the cleaned data.

In [ ]:
# scatterplot: does lot size tend to shrink as floors increase?
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x="floors", y="sqft_lot", alpha=0.3)
plt.title("Floors vs. Lot Size (Urban Houses)")
plt.xlabel("Floors")
plt.ylabel("Lot Size (sqft)")
plt.show()

In [ ]:
# a boxplot is a good complement since floors is a discrete value
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x="floors", y="sqft_lot")
plt.title("Lot Size by Floor Count (Urban Houses)")
plt.xlabel("Floors")
plt.ylabel("Lot Size (sqft)")
plt.show()

In [ ]:
# Spearman correlation between floors and lot size
# a negative value means: more floors tends to go together with a smaller lot
corr, p_value = spearmanr(df_clean["floors"], df_clean["sqft_lot"])

print(f"Spearman correlation: {corr:.3f}")
print(f"p-value: {p_value:.5f}")

Both plots suggest that houses with more floors tend to have smaller lots. To check this more formally, we use a **Spearman correlation**. It measures whether two variables move together in a consistent direction (up or down), which fits `floors` well since it only has a few discrete values (1, 1.5, 2, 2.5, 3, 3.5).

**How to read this:**
- The correlation is between -1 and 1. A negative number close to -1 means a strong tendency for lot size to shrink as floors increase, which would support H3.
- The p-value tells us whether this result is likely just random noise. A p-value below 0.05 means the relationship is unlikely to be a coincidence.

Compare the printed numbers above to these rules of thumb to decide whether the data supports H3 for William's city house search.

## 5. Save the Cleaned Data

We export `df_clean` (Urban houses, IQR-cleaned, deduplicated) to `../data/processed/df_h3_cleaned.csv` so it can be reused without re-running this notebook.

In [ ]:
# save the cleaned Urban dataset for reuse
df_clean.to_csv("../data/processed/df_h3_cleaned.csv", index=False)
print(f"Saved {len(df_clean):,} rows to ../data/processed/df_h3_cleaned.csv")

## 6. Suggested Properties for William

Based on the H3 result (more floors go together with smaller lots for Urban houses), we suggest 2 properties that best fit William's "compact, vertical city house" profile:

- **Urban** (already guaranteed - `df_clean` only contains Urban houses)
- **Floors ≥ 2** - a genuinely multi-story, vertical layout
- **`sqft_lot` at or below the Urban median** - a compact lot rather than an average-or-larger one
- **Price** - William wants to buy fast. We assume (per his request) that more expensive houses tend to sell faster, so among the matches we now rank by **price descending** and take the 2 most expensive - trading off "price-sensitive" for "sells quickly".

Note: this "expensive houses sell faster" assumption is not verified against the data (there is no days-on-market or listing-date column in this dataset to check it against) - it is applied here as a stated business rule from William.

In [ ]:
# criteria: multi-story (>= 2 floors) and a compact lot (<= Urban median sqft_lot);
# ranked by price DESCENDING, since William wants to buy fast and more
# expensive houses are assumed to sell faster (his stated business rule)
median_lot = df_clean["sqft_lot"].median()

candidates = df_clean[
    (df_clean["floors"] >= 2) & (df_clean["sqft_lot"] <= median_lot)
].sort_values("price", ascending=False)

top2 = candidates.head(2)

print(f"{len(candidates):,} houses match William's criteria (floors >= 2, sqft_lot <= {median_lot:,.0f}).")
print("Top 2 suggestions (most expensive among the matches, assumed to sell fastest):\n")

top2[["id", "price", "floors", "sqft_lot", "sqft_living", "bedrooms", "bathrooms", "zipcode"]]